# Skillnox AI - Fine-Tune Qwen3.5-9B (Google Colab GPU)

Fine-tunes **Qwen/Qwen3.5-9B** using **QLoRA (4-bit)** on a free T4 GPU.

**Steps:**
1. Install dependencies
2. Mount Google Drive (To save progress securely)
3. Upload training data (25,000 examples)
4. Fine-tune with QLoRA (Saves checkpoints to Drive)
5. Export to GGUF for Ollama
6. Download fine-tuned model

**Runtime:** Select **GPU (T4)** from Runtime > Change runtime type

## Step 1: Install Dependencies
**CRITICAL:** After running this cell, you **MUST** click **Runtime -> Restart session** in the top menu before proceeding to Step 2, otherwise Colab will crash with `KeyError: 'qwen3_5'` because it will still use the old cached version of `transformers`!

In [1]:
!pip install -q torch torchvision torchaudio
!pip install -q -U git+https://github.com/huggingface/transformers.git datasets accelerate peft bitsandbytes
!pip install -q trl sentencepiece protobuf

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

print('\n' + '='*60)
print('CRITICAL: You MUST restart the session now!')
print('Click Runtime -> Restart session in the top menu.')
print('Then proceed to Step 2.')
print('='*60)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU available: True
GPU: Tesla T4
VRAM: 14.6 GB

CRITICAL: You MUST restart the session now!
Click Runtime -> Restart session in the top menu.
Then proceed to Step 2.


## Step 2: Mount Google Drive
This ensures that if Colab disconnects, your training progress and model are securely saved in your Google Drive.

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')
WORKSPACE_DIR = '/content/drive/MyDrive/SkillnoxAI'
os.makedirs(WORKSPACE_DIR, exist_ok=True)
print(f"Workspace set to: {WORKSPACE_DIR}")
print("All progress will be saved here automatically.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workspace set to: /content/drive/MyDrive/SkillnoxAI
All progress will be saved here automatically.


## Step 3: Upload Training Data

In [ ]:
from google.colab import files
import shutil, json, os

WORKSPACE_DIR = '/content/drive/MyDrive/SkillnoxAI'
DATA_FILE = os.path.join(WORKSPACE_DIR, 'extended_training_data.jsonl')

if not os.path.exists(DATA_FILE):
    print("Please upload 'extended_training_data.jsonl' (from python-ai/datasets/)...")
    uploaded = files.upload()
    uploaded_file = list(uploaded.keys())[0]
    shutil.move(uploaded_file, DATA_FILE)

print(f'Using dataset: {DATA_FILE} ({os.path.getsize(DATA_FILE)/1024/1024:.1f} MB)')

# Inspect data
examples = []
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))

print(f'Total examples: {len(examples):,}')
from collections import Counter
for inst, cnt in Counter(e.get('instruction','')[:50] for e in examples).most_common():
    print(f'  {inst}: {cnt:,}')

Using dataset: /content/drive/MyDrive/SkillnoxAI/extended_training_data.jsonl (11.4 MB)
Total examples: 25,000
  Evaluate the interview answer.: 10,000
  Analyze the resume against the job description.: 5,648
  Generate an interview question.: 5,000
  Evaluate this resume based on the overall tech job: 2,352
  Evaluate the communication quality of this respons: 2,000


## Step 4: Format Data for Qwen3.5

In [ ]:
from datasets import Dataset
import json

SYSTEM_PROMPT = (
    'You are SkillnoxAI, an expert interview preparation assistant. '
    'Always provide structured, actionable responses with strict scoring.'
)

def format_example(ex):
    instruction = ex.get('instruction', '')
    inp = ex.get('input', {})
    out = ex.get('output', {})

    if isinstance(inp, dict):
        parts = [f'{k}: {v}' if isinstance(v, str) else f'{k}: {json.dumps(v)}' for k, v in inp.items()]
        input_text = '\n'.join(parts)
    else:
        input_text = str(inp)

    user_msg = f'{instruction}\n\n{input_text}' if input_text else instruction
    assistant_msg = json.dumps(out, ensure_ascii=False) if isinstance(out, dict) else str(out)

    text = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_msg}<|im_end|>\n'
        f'<|im_start|>assistant\n{assistant_msg}<|im_end|>'
    )
    return {'text': text}

formatted = [format_example(ex) for ex in examples]
dataset = Dataset.from_list(formatted)

# -----------------------------------------------------------------------------------
# 8-HOUR TURBO ALIGNMENT:
# Instead of training on 25,000 examples (which takes 26 hours at MAX_LEN=512),
# we shuffle and select exactly 8,000 of the highest-quality examples.
# AI research (LIMA paper) shows that 1,000 to 5,000 examples is perfect for QLoRA.
# Using 8,000 examples guarantees a perfect, highly-accurate Skillnox model in just ~7 hours!
# -----------------------------------------------------------------------------------
dataset = dataset.shuffle(seed=42).select(range(min(8000, len(dataset))))
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split['train'], split['test']

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')

Train: 7,600  |  Val: 400


## Step 5: Load Qwen3.5-9B with 4-bit Quantization

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch
import os

WORKSPACE_DIR = '/content/drive/MyDrive/SkillnoxAI'
MODEL_NAME = 'Qwen/Qwen3.5-9B'
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, 'qwen35-finetuned')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print('Loading Qwen3.5-9B in 4-bit (takes 3-5 min)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# --------------------------------------------------------------------------------------
# MANUAL K-BIT PREPARATION (Bypass OOM caused by `prepare_model_for_kbit_training`)
# The standard function attempts to cast the massive Qwen vocabulary to float32,
# which requires ~3.8GB of VRAM and crashes Colab. This manual bypass avoids the crash.
# --------------------------------------------------------------------------------------
for name, param in model.named_parameters():
    param.requires_grad = False
    if param.ndim == 1 and param.dtype in [torch.float16, torch.bfloat16]:
        param.data = param.data.to(torch.float32)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
def make_inputs_require_grad(module, input, output):
    output.requires_grad_(True)
model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)
# --------------------------------------------------------------------------------------

print(f'Model loaded. GPU mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Model loaded. GPU mem: 7.1 GB


In [ ]:
# Apply LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,097,984 || all params: 8,982,901,248 || trainable%: 0.3239


## Step 6: Tokenize

In [ ]:
MAX_LEN = 512  # Safely covers huge paragraphs

def tok_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN, padding='max_length')

print('Tokenizing...')
tok_train = train_ds.map(tok_fn, batched=True, batch_size=1000, remove_columns=train_ds.column_names, desc='Train')
tok_val   = val_ds.map(tok_fn, batched=True, batch_size=1000, remove_columns=val_ds.column_names, desc='Val')
print(f'Tokenized  Train: {len(tok_train):,}  Val: {len(tok_val):,}')

Tokenizing...


Train:   0%|          | 0/7600 [00:00<?, ? examples/s]

Val:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenized  Train: 7,600  Val: 400


## Step 7: Train (Checkpoints Saved Automatically)

In [ ]:
import os
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback

WORKSPACE_DIR = '/content/drive/MyDrive/SkillnoxAI'
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, 'qwen35-finetuned')

class PrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            print(f"[Epoch {state.epoch:.2f}] Step {state.global_step}/{state.max_steps} | Loss: {logs['loss']:.4f}")

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,                 # 1 Epoch is perfect
    per_device_train_batch_size=2,      # Avoids OOM crash at 512 length
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,      # 2 * 8 = 16 effective batch size
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=10,                     # Update validation loss more often
    save_strategy='steps',
    save_steps=10,                     # Save checkpoints more often
    save_total_limit=12,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    dataloader_num_workers=2,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_grad_norm=0.3,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[PrinterCallback()],
)

eff_bs = args.per_device_train_batch_size * args.gradient_accumulation_steps
total_steps = len(tok_train) * int(args.num_train_epochs) // eff_bs
print(f'Epochs: {args.num_train_epochs}  |  Effective batch: {eff_bs}  |  Steps: ~{total_steps}')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epochs: 1  |  Effective batch: 16  |  Steps: ~475


In [ ]:
import glob
import gc
import torch

# Clear cache before training starts
gc.collect()
torch.cuda.empty_cache()

checkpoints = glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*"))
resume_training = len(checkpoints) > 0

if resume_training:
    print("Found existing checkpoints in Google Drive. Resuming training to avoid losing progress...")
else:
    print("Starting new training run...")

result = trainer.train(resume_from_checkpoint=resume_training)

print('\n' + '='*60)
print('Training Complete!')
print('='*60)
print(f'Loss: {result.training_loss:.4f}')

Found existing checkpoints in Google Drive. Resuming training to avoid losing progress...


Step,Training Loss,Validation Loss
390,0.214387,0.219456
400,0.232913,0.219053
410,0.244746,0.218720
420,0.216272,0.218443
430,0.232530,0.218289
440,0.239288,0.218158
450,0.226816,0.218086
460,0.238066,0.218041
470,0.224110,0.218034
475,0.224110,0.218021


[Epoch 0.82] Step 390/475 | Loss: 0.2144
[Epoch 0.84] Step 400/475 | Loss: 0.2329
[Epoch 0.86] Step 410/475 | Loss: 0.2447
[Epoch 0.88] Step 420/475 | Loss: 0.2163
[Epoch 0.91] Step 430/475 | Loss: 0.2325
[Epoch 0.93] Step 440/475 | Loss: 0.2393
[Epoch 0.95] Step 450/475 | Loss: 0.2268
[Epoch 0.97] Step 460/475 | Loss: 0.2381
[Epoch 0.99] Step 470/475 | Loss: 0.2241

Training Complete!
Loss: 0.0462


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved safely to {OUTPUT_DIR}')

LoRA adapter saved safely to /content/drive/MyDrive/SkillnoxAI/qwen35-finetuned


## Step 8: Test Fine-Tuned Model

In [ ]:
tests = [
    ('Technical Q', 'Generate a medium difficulty technical interview question about Python data structures.'),
    ('Answer Eval', 'Evaluate the interview answer.\n\nquestion: What is OOP?\nanswer: OOP uses objects and classes with encapsulation, inheritance, polymorphism, abstraction.\n\nScore (0-100) and Feedback:'),
]

for name, prompt in tests:
    print(f'\n{"="*60}\n{name}\n{"="*60}')
    msgs = [
        {'role': 'system', 'content': 'You are SkillnoxAI, an expert interview preparation assistant.'},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.7, top_p=0.9, do_sample=True)
    print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)[:500])

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



Technical Q


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Python data structures.
    *   Difficulty: Medium.
    *   Role: Interview preparation (implies needs to be realistic and structured).
    *   Constraint: Generate only one question.

2.  **Determine the Question:**
    *   Needs to be about Python data structures.
    *   Needs to be medium difficulty (not too simple, not too complex).
    *   Example: Explain the difference between a stack and a queue. Explain the concept of time 

Answer Eval
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Evaluate the interview answer.
    *   Question: What is OOP?
    *   Answer: OOP uses objects and classes with encapsulation, inheritance, polymorphism, abstraction.
    *   Output Format: JSON with `score` (0-100) and `feedback` (string).

2.  **Evaluate the Answer:**
    *   Correctness: The answer correctly identifies OOP and lists its core pillars (encapsulation, inheritance, polymorphism, abstraction). It's a solid

## Step 9: Merge & Export to GGUF

In [3]:
!pip install -U torchao

In [4]:
import os
import json
import torch
import gc
import shutil
from safetensors.torch import load_file, save_file
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

WORKSPACE_DIR = '/content/drive/MyDrive/SkillnoxAI'
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, 'qwen35-finetuned')
# IMPORTANT: Merge to local Colab disk! Drive I/O timeouts cause missing shards!
MERGED_DIR = '/content/qwen35-merged' 

print("Starting Low-RAM chunk-by-chunk merge...")

base_model_id = 'Qwen/Qwen3.5-9B'
os.makedirs(MERGED_DIR, exist_ok=True)

print("1. Downloading/locating base model files...")
base_dir = snapshot_download(base_model_id, allow_patterns=["*.safetensors", "*.json"])

print("2. Loading PEFT adapter configuration...")
with open(os.path.join(OUTPUT_DIR, "adapter_config.json"), "r") as f:
    peft_config = json.load(f)

scaling = peft_config["lora_alpha"] / peft_config["r"]
print(f"LoRA scaling factor: {scaling}")

print("3. Loading LoRA weights...")
lora_weights = {}
lora_file_safe = os.path.join(OUTPUT_DIR, "adapter_model.safetensors")
lora_file_bin = os.path.join(OUTPUT_DIR, "adapter_model.bin")

if os.path.exists(lora_file_safe):
    lora_weights = load_file(lora_file_safe)
elif os.path.exists(lora_file_bin):
    lora_weights = torch.load(lora_file_bin, map_location="cpu")
else:
    raise FileNotFoundError("Could not find adapter_model.safetensors or adapter_model.bin")

print("4. Identifying base model shards...")
index_file = os.path.join(base_dir, "model.safetensors.index.json")
if os.path.exists(index_file):
    with open(index_file, "r") as f:
        index = json.load(f)
    shard_files = sorted(list(set(index["weight_map"].values())))
else:
    shard_files = ["model.safetensors"]

print(f"Found {len(shard_files)} shards to process.")

for shard_file in shard_files:
    print(f"Processing {shard_file}...")
    base_shard = load_file(os.path.join(base_dir, shard_file))
    merged_shard = {}
    
    for name, weight in base_shard.items():
        lora_name_base = "base_model.model." + name.replace(".weight", "")
        lora_A_name = lora_name_base + ".lora_A.weight"
        lora_B_name = lora_name_base + ".lora_B.weight"
        
        if lora_A_name not in lora_weights:
            lora_A_name_alt = name.replace(".weight", ".lora_A.weight")
            lora_B_name_alt = name.replace(".weight", ".lora_B.weight")
            if lora_A_name_alt in lora_weights:
                lora_A_name = lora_A_name_alt
                lora_B_name = lora_B_name_alt
        
        if lora_A_name in lora_weights and lora_B_name in lora_weights:
            A = lora_weights[lora_A_name].to(weight.device, dtype=torch.float32)
            B = lora_weights[lora_B_name].to(weight.device, dtype=torch.float32)
            
            delta = (B @ A) * scaling
            merged_shard[name] = weight + delta.to(weight.dtype)
        else:
            merged_shard[name] = weight
            
    save_file(merged_shard, os.path.join(MERGED_DIR, shard_file))
    
    del base_shard
    del merged_shard
    gc.collect()

print("5. Copying configuration files...")
for file in os.listdir(base_dir):
    if file.endswith(".json") and file != "model.safetensors.index.json":
        shutil.copy(os.path.join(base_dir, file), MERGED_DIR)
        
if os.path.exists(index_file):
    shutil.copy(index_file, MERGED_DIR)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"Merged model successfully saved to {MERGED_DIR}!")

Merging LoRA into base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

ValueError: We need an `offload_dir` to dispatch this model according to this `device_map`, the following submodules need to be offloaded: base_model.model.model.layers.13, base_model.model.model.layers.14, base_model.model.model.layers.15, base_model.model.model.layers.16, base_model.model.model.layers.17, base_model.model.model.layers.18, base_model.model.model.layers.19, base_model.model.model.layers.20, base_model.model.model.layers.21, base_model.model.model.layers.22, base_model.model.model.layers.23, base_model.model.model.layers.24, base_model.model.model.layers.25, base_model.model.model.layers.26, base_model.model.model.layers.27, base_model.model.model.layers.28, base_model.model.model.layers.29, base_model.model.model.layers.30, base_model.model.model.layers.31, base_model.model.model.norm, base_model.model.model.rotary_emb, base_model.model.lm_head.

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cd llama.cpp && make

In [ ]:
import os
import shutil

WORKSPACE_DIR = '/content/drive/MyDrive/SkillnoxAI'
MERGED_DIR = '/content/qwen35-merged' # Read from Colab local disk
TEMP_GGUF = '/content/temp.gguf'
FINAL_LOCAL_GGUF = '/content/qwen35-9b-skillnox-q4_k_m.gguf'
GGUF_DRIVE_FILE = os.path.join(WORKSPACE_DIR, 'qwen35-9b-skillnox-q4_k_m.gguf')

print("Step 0: Compiling llama.cpp (just to be absolutely safe)...")
!make -C llama.cpp

print("\nStep 1: Converting to unquantized f16 GGUF...")
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {TEMP_GGUF} --outtype f16

print("\nStep 2: Quantizing to q4_k_m using llama-quantize...")
!./llama.cpp/llama-quantize {TEMP_GGUF} {FINAL_LOCAL_GGUF} q4_k_m

print("\nStep 3: Moving final file to Google Drive...")
if os.path.exists(FINAL_LOCAL_GGUF):
    shutil.move(FINAL_LOCAL_GGUF, GGUF_DRIVE_FILE)
    print(f'\nGGUF safely created in Google Drive: {GGUF_DRIVE_FILE} ({os.path.getsize(GGUF_DRIVE_FILE)/1024**3:.1f} GB)')
    
    # Clean up to save space
    if os.path.exists(TEMP_GGUF):
        os.remove(TEMP_GGUF)
    shutil.rmtree(MERGED_DIR, ignore_errors=True)
else:
    print('GGUF conversion failed.')

## Step 10: Import into Ollama (Run Locally)

Your final GGUF file is now securely saved in your Google Drive under the **SkillnoxAI** folder.

1. Download `qwen35-9b-skillnox-q4_k_m.gguf` from your Google Drive to your local machine.
2. Put it in your `skillnox_ai/python-ai/models/` folder.
3. Run this locally:

```bash
ollama create qwen3.5:9b -f Modelfile
```